In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

BASE = {
    "dosis_g": 18.0,
    "temp_c": 96.0,
    "presion_bar": 19.5,
    "tiempo_s": 28.0,
    "rendimiento_g": 45.0,
}

ANALISIS_TABLAS = {
    "tabla_1_dosis": (
        "Al aumentar la dosis de 15.0 g a 21.0 g manteniendo 45.0 g en taza, "
        "el ratio baja (de 3.00 a 2.14) y el lecho ofrece mayor resistencia al "
        "flujo, aumentando el tiempo estimado. Con dosis bajas el agua pasa más "
        "rápido y se corre riesgo de subextracción; con dosis altas la extracción "
        "por gramo de café se reduce y el perfil se concentra más."
    ),
    "tabla_2_ratio": (
        "Al variar el rendimiento en taza de 30.0 g a 63.0 g con 18.0 g de dosis, "
        "sube directamente el ratio (de 1.67 a 3.50) y el tiempo de contacto. "
        "Ratios cortos y tiempos bajos favorecen la subextracción, mientras que "
        "ratios largos y tiempos altos (>34 s) aumentan el riesgo de "
        "sobreextracción y notas amargas o astringentes."
    ),
    "tabla_3_temperatura": (
        "Variar la temperatura entre 90 °C y 99 °C respecto a 96 °C modifica la "
        "cinética de extracción y la viscosidad del agua. Temperaturas más bajas "
        "reducen la extracción efectiva (riesgo de subextracción), mientras que "
        "temperaturas de 98–99 °C aceleran la disolución y elevan el riesgo de "
        "sobreextracción."
    ),
    "tabla_4_presion": (
        "Al mover la presión de 16.0 a 22.0 bar frente al rango de referencia de "
        "19–20 bar (base 19.5 bar), cambian el flujo y el esfuerzo sobre el "
        "lecho de café. Presiones bajas pueden quedarse cortas y presiones muy "
        "altas (22.0 bar) pueden favorecer canalizaciones o extracciones "
        "desiguales si la molienda no acompaña."
    ),
}


def calcular_metricas(dosis_g, temp_c, presion_bar, tiempo_s, rendimiento_g):
  ratio = rendimiento_g / dosis_g
  caudal_g_s = rendimiento_g / tiempo_s
  ey_sim = (
      18.0
      * ((ratio / 2.5) ** 0.35)
      * ((temp_c / 96.0) ** 0.60)
      * ((tiempo_s / 28.0) ** 0.25)
      * (1.0 - 0.015 * max(0.0, presion_bar - 20.0))
  )

  if tiempo_s < 23 or ratio < 2.0 or ey_sim < 16.5:
    estado = "Subextracción / flujo rápido"
  elif tiempo_s > 34 or ratio > 3.2 or ey_sim > 20.5:
    estado = "Sobreextracción / flujo lento"
  else:
    estado = "Zona equilibrada"

  return round(ratio, 2), round(caudal_g_s, 2), round(ey_sim, 2), estado


def mostrar_tabla_con_analisis(titulo, df, clave_analisis):
  print(f"\n=== {titulo} ===")
  display(df)
  print("Análisis:", ANALISIS_TABLAS[clave_analisis])


# Tabla 1: Variación de dosis
filas_t1 = []
for d in [15.0, 16.5, 18.0, 19.5, 21.0]:
  t_est = round(BASE["tiempo_s"] * ((d / BASE["dosis_g"]) ** 0.45), 1)
  ratio, caudal, ey, estado = calcular_metricas(
      d, BASE["temp_c"], BASE["presion_bar"], t_est, BASE["rendimiento_g"]
  )
  filas_t1.append({
      "Dosis (g)": d,
      "Rendimiento (g)": BASE["rendimiento_g"],
      "Ratio (g/g)": ratio,
      "Tiempo est. (s)": t_est,
      "Caudal (g/s)": caudal,
      "EY sim (%)": ey,
      "Diagnóstico": estado,
  })
tabla_1_dosis = pd.DataFrame(filas_t1)

# Tabla 2: Variación de rendimiento / ratio
filas_t2 = []
for r in [30.0, 36.0, 45.0, 54.0, 63.0]:
  t_est = round(BASE["tiempo_s"] * (r / BASE["rendimiento_g"]), 1)
  ratio, caudal, ey, estado = calcular_metricas(
      BASE["dosis_g"], BASE["temp_c"], BASE["presion_bar"], t_est, r
  )
  filas_t2.append({
      "Dosis (g)": BASE["dosis_g"],
      "Rendimiento (g)": r,
      "Ratio (g/g)": ratio,
      "Tiempo est. (s)": t_est,
      "Caudal (g/s)": caudal,
      "EY sim (%)": ey,
      "Diagnóstico": estado,
  })
tabla_2_ratio = pd.DataFrame(filas_t2)

# Tabla 3: Variación de temperatura
filas_t3 = []
for temp in [90.0, 93.0, 96.0, 98.0, 99.0]:
  t_est = round(BASE["tiempo_s"] * ((96.0 / temp) ** 0.30), 1)
  ratio, caudal, ey, estado = calcular_metricas(
      BASE["dosis_g"], temp, BASE["presion_bar"], t_est, BASE["rendimiento_g"]
  )
  filas_t3.append({
      "Temp. agua (°C)": temp,
      "Tiempo est. (s)": t_est,
      "Caudal (g/s)": caudal,
      "EY sim (%)": ey,
      "Diagnóstico": estado,
  })
tabla_3_temperatura = pd.DataFrame(filas_t3)

# Tabla 4: Variación de presión
filas_t4 = []
for p in [16.0, 18.0, 19.5, 20.0, 22.0]:
  t_est = round(BASE["tiempo_s"] * (1.0 + 0.015 * (p - BASE["presion_bar"])), 1)
  ratio, caudal, ey, estado = calcular_metricas(
      BASE["dosis_g"], BASE["temp_c"], p, t_est, BASE["rendimiento_g"]
  )
  filas_t4.append({
      "Presión (bar)": p,
      "Rango ref. (19-20)": "Dentro" if 19.0 <= p <= 20.0 else "Fuera",
      "Tiempo est. (s)": t_est,
      "Caudal (g/s)": caudal,
      "EY sim (%)": ey,
      "Diagnóstico": estado,
  })
tabla_4_presion = pd.DataFrame(filas_t4)

mostrar_tabla_con_analisis(
    "TABLA 1: VARIACIÓN DE DOSIS", tabla_1_dosis, "tabla_1_dosis"
)
mostrar_tabla_con_analisis(
    "TABLA 2: VARIACIÓN DE RENDIMIENTO / RATIO", tabla_2_ratio, "tabla_2_ratio"
)
mostrar_tabla_con_analisis(
    "TABLA 3: VARIACIÓN DE TEMPERATURA",
    tabla_3_temperatura,
    "tabla_3_temperatura",
)
mostrar_tabla_con_analisis(
    "TABLA 4: VARIACIÓN DE PRESIÓN", tabla_4_presion, "tabla_4_presion"
)


=== TABLA 1: VARIACIÓN DE DOSIS ===


,Dosis (g),Rendimiento (g),Ratio (g/g),Tiempo est. (s),Caudal (g/s),EY sim (%),Diagnóstico
0,15.0,45.0,3.00,25.8,1.74,18.80,Zona equilibrada
1,16.5,45.0,2.73,26.9,1.67,18.37,Zona equilibrada
2,18.0,45.0,2.50,28.0,1.61,18.00,Zona equilibrada
3,19.5,45.0,2.31,29.0,1.55,17.66,Zona equilibrada
4,21.0,45.0,2.14,30.0,1.50,17.35,Zona equilibrada


Análisis: Al aumentar la dosis de 15.0 g a 21.0 g manteniendo 45.0 g en taza, el ratio baja (de 3.00 a 2.14) y el lecho ofrece mayor resistencia al flujo, aumentando el tiempo estimado. Con dosis bajas el agua pasa más rápido y se corre riesgo de subextracción; con dosis altas la extracción por gramo de café se reduce y el perfil se concentra más.

=== TABLA 2: VARIACIÓN DE RENDIMIENTO / RATIO ===


,Dosis (g),Rendimiento (g),Ratio (g/g),Tiempo est. (s),Caudal (g/s),EY sim (%),Diagnóstico
0,18.0,30.0,1.67,18.7,1.60,14.12,Subextracción / flujo rápido
1,18.0,36.0,2.00,22.4,1.61,15.74,Subextracción / flujo rápido
2,18.0,45.0,2.50,28.0,1.61,18.00,Zona equilibrada
3,18.0,54.0,3.00,33.6,1.61,20.08,Zona equilibrada
4,18.0,63.0,3.50,39.2,1.61,22.03,Sobreextracción / flujo lento


Análisis: Al variar el rendimiento en taza de 30.0 g a 63.0 g con 18.0 g de dosis, sube directamente el ratio (de 1.67 a 3.50) y el tiempo de contacto. Ratios cortos y tiempos bajos favorecen la subextracción, mientras que ratios largos y tiempos altos (>34 s) aumentan el riesgo de sobreextracción y notas amargas o astringentes.

=== TABLA 3: VARIACIÓN DE TEMPERATURA ===


,Temp. agua (°C),Tiempo est. (s),Caudal (g/s),EY sim (%),Diagnóstico
0,90.0,28.5,1.58,17.39,Zona equilibrada
1,93.0,28.3,1.59,17.71,Zona equilibrada
2,96.0,28.0,1.61,18.00,Zona equilibrada
3,98.0,27.8,1.62,18.19,Zona equilibrada
4,99.0,27.7,1.62,18.29,Zona equilibrada


Análisis: Variar la temperatura entre 90 °C y 99 °C respecto a 96 °C modifica la cinética de extracción y la viscosidad del agua. Temperaturas más bajas reducen la extracción efectiva (riesgo de subextracción), mientras que temperaturas de 98–99 °C aceleran la disolución y elevan el riesgo de sobreextracción.

=== TABLA 4: VARIACIÓN DE PRESIÓN ===


,Presión (bar),Rango ref. (19-20),Tiempo est. (s),Caudal (g/s),EY sim (%),Diagnóstico
0,16.0,Fuera,26.5,1.70,17.75,Zona equilibrada
1,18.0,Fuera,27.4,1.64,17.90,Zona equilibrada
2,19.5,Dentro,28.0,1.61,18.00,Zona equilibrada
3,20.0,Dentro,28.2,1.60,18.03,Zona equilibrada
4,22.0,Fuera,29.1,1.55,17.63,Zona equilibrada


Análisis: Al mover la presión de 16.0 a 22.0 bar frente al rango de referencia de 19–20 bar (base 19.5 bar), cambian el flujo y el esfuerzo sobre el lecho de café. Presiones bajas pueden quedarse cortas y presiones muy altas (22.0 bar) pueden favorecer canalizaciones o extracciones desiguales si la molienda no acompaña.


### TABLA 1: VARIACIÓN DE DOSIS

| Fila | Dosis (g) | Rendimiento (g) | Ratio (g/g) | Tiempo est. (s) | Caudal (g/s) | EY sim (%) | Diagnóstico |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| 0 | 15.0 | 45.0 | 3.00 | 25.8 | 1.74 | 18.80 | Zona equilibrada |
| 1 | 16.5 | 45.0 | 2.73 | 26.9 | 1.67 | 18.37 | Zona equilibrada |
| 2 | 18.0 | 45.0 | 2.50 | 28.0 | 1.61 | 18.00 | Zona equilibrada |
| 3 | 19.5 | 45.0 | 2.31 | 29.0 | 1.55 | 17.66 | Zona equilibrada |
| 4 | 21.0 | 45.0 | 2.14 | 30.0 | 1.50 | 17.35 | Zona equilibrada |

### TABLA 2: VARIACIÓN DE RENDIMIENTO / RATIO

| Fila | Dosis (g) | Rendimiento (g) | Ratio (g/g) | Tiempo est. (s) | Caudal (g/s) | EY sim (%) | Diagnóstico |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| 0 | 18.0 | 30.0 | 1.67 | 18.7 | 1.60 | 14.12 | Subextracción / flujo rápido |
| 1 | 18.0 | 36.0 | 2.00 | 22.4 | 1.61 | 15.74 | Subextracción / flujo rápido |
| 2 | 18.0 | 45.0 | 2.50 | 28.0 | 1.61 | 18.00 | Zona equilibrada |
| 3 | 18.0 | 54.0 | 3.00 | 33.6 | 1.61 | 20.08 | Zona equilibrada |
| 4 | 18.0 | 63.0 | 3.50 | 39.2 | 1.61 | 22.03 | Sobreextracción / flujo lento |

### TABLA 3: VARIACIÓN DE TEMPERATURA

| Fila | Temp. agua (°C) | Tiempo est. (s) | Caudal (g/s) | EY sim (%) | Diagnóstico |
| :---: | :---: | :---: | :---: | :---: | :--- |
| 0 | 90.0 | 28.5 | 1.58 | 17.39 | Zona equilibrada |
| 1 | 93.0 | 28.3 | 1.59 | 17.71 | Zona equilibrada |
| 2 | 96.0 | 28.0 | 1.61 | 18.00 | Zona equilibrada |
| 3 | 98.0 | 27.8 | 1.62 | 18.19 | Zona equilibrada |
| 4 | 99.0 | 27.7 | 1.62 | 18.29 | Zona equilibrada |

### TABLA 4: VARIACIÓN DE PRESIÓN

| Fila | Presión (bar) | Rango ref. (19-20) | Tiempo est. (s) | Caudal (g/s) | EY sim (%) | Diagnóstico |
| :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| 0 | 16.0 | Fuera | 26.5 | 1.70 | 17.75 | Zona equilibrada |
| 1 | 18.0 | Fuera | 27.4 | 1.64 | 17.90 | Zona equilibrada |
| 2 | 19.5 | Dentro | 28.0 | 1.61 | 18.00 | Zona equilibrada |
| 3 | 20.0 | Dentro | 28.2 | 1.60 | 18.05 | Zona equilibrada |